# Haiku를 서브에이전트로 사용하기

이 레시피에서는 Claude 3 Haiku 서브에이전트 모델로 실적 발표 PDF에서 관련 정보를 추출해 Apple의 2023 회계연도 실적 보고서를 분석해 봅니다. 그런 다음 Claude 3 Opus로 질문에 대한 답을 생성하고, matplotlib으로 답변에 곁들일 그래프를 만듭니다.

## 1단계: 환경 설정
먼저 필요한 라이브러리를 설치하고 Claude API 클라이언트를 설정합니다.

In [ ]:
%pip install anthropic IPython PyMuPDF matplotlib

In [89]:
# Import the required libraries
import base64
import io
import os
from concurrent.futures import ThreadPoolExecutor

import fitz
import requests
from anthropic import Anthropic
from PIL import Image

# Set up the Claude API client
client = Anthropic()
MODEL_NAME = "claude-haiku-4-5"

## 2단계: 문서 모으고 질문 던지기
이 예제에서는 Apple의 2023 회계연도 재무제표 전부를 사용해 그해의 순매출에 대해 질문합니다.

In [90]:
# List of Apple's earnings release PDF URLs
pdf_urls = [
    "https://www.apple.com/newsroom/pdfs/fy2023-q4/FY23_Q4_Consolidated_Financial_Statements.pdf",
    "https://www.apple.com/newsroom/pdfs/fy2023-q3/FY23_Q3_Consolidated_Financial_Statements.pdf",
    "https://www.apple.com/newsroom/pdfs/FY23_Q2_Consolidated_Financial_Statements.pdf",
    "https://www.apple.com/newsroom/pdfs/FY23_Q1_Consolidated_Financial_Statements.pdf",
]

# User's question
QUESTION = "How did Apple's net sales change quarter to quarter in the 2023 financial year and what were the key contributors to the changes?"

## 3단계: PDF 내려받아 이미지로 변환하기
다음으로 실적 발표 PDF를 내려받아 base64로 인코딩된 PNG 이미지로 변환하는 함수를 정의합니다. 이 PDF들은 전통적인 PDF 파서로 파싱하기 어려운 표로 가득 차 있어서 이렇게 해야 합니다. 그냥 이미지로 변환해 Haiku에 전달하는 편이 더 쉽습니다.

```download_pdf``` 함수는 주어진 URL에서 PDF 파일을 내려받아 지정한 폴더에 저장합니다. ```pdf_to_base64_pngs``` 함수는 PDF를 base64로 인코딩된 PNG 이미지 목록으로 변환합니다.

In [ ]:
# Function to download a PDF file from a URL and save it to a specified folder
def download_pdf(url, folder):
    response = requests.get(url, timeout=60)
    if response.status_code == 200:
        file_name = os.path.join(folder, url.split("/")[-1])
        with open(file_name, "wb") as file:
            file.write(response.content)
        return file_name
    else:
        print(f"Failed to download PDF from {url}")
        return None


# Define the function to convert a PDF to a list of base64-encoded PNG images
def pdf_to_base64_pngs(pdf_path, quality=75, max_size=(1024, 1024)):
    # Open the PDF file
    doc = fitz.open(pdf_path)

    base64_encoded_pngs = []

    # Iterate through each page of the PDF
    for page_num in range(doc.page_count):
        # Load the page
        page = doc.load_page(page_num)

        # Render the page as a PNG image
        pix = page.get_pixmap(matrix=fitz.Matrix(300 / 72, 300 / 72))

        # Convert the pixmap to a PIL Image
        image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        # Resize the image if it exceeds the maximum size
        if image.size[0] > max_size[0] or image.size[1] > max_size[1]:
            image.thumbnail(max_size, Image.Resampling.LANCZOS)

        # Convert the PIL Image to base64-encoded PNG
        image_data = io.BytesIO()
        image.save(image_data, format="PNG", optimize=True, quality=quality)
        image_data.seek(0)
        base64_encoded = base64.b64encode(image_data.getvalue()).decode("utf-8")
        base64_encoded_pngs.append(base64_encoded)

    # Close the PDF document
    doc.close()

    return base64_encoded_pngs


# Folder to save the downloaded PDFs
folder = "../images/using_sub_agents"


# Create the directory if it doesn't exist
os.makedirs(folder)

# Download the PDFs concurrently
with ThreadPoolExecutor() as executor:
    pdf_paths = list(executor.map(download_pdf, pdf_urls, [folder] * len(pdf_urls)))

# Remove any None values (failed downloads) from pdf_paths
pdf_paths = [path for path in pdf_paths if path is not None]

ThreadPoolExecutor로 PDF를 동시에 내려받고 파일 경로를 pdf_paths에 저장합니다.

## 4단계: Opus로 Haiku용 맞춤 프롬프트 만들기
Opus를 오케스트레이터로 삼아, 사용자가 준 질문을 바탕으로 각 Haiku 서브에이전트를 위한 맞춤 프롬프트를 작성하게 해 보겠습니다.

In [102]:
def generate_haiku_prompt(question):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"Based on the following question, please generate a specific prompt for an LLM sub-agent to extract relevant information from an earning's report PDF. Each sub-agent only has access to a single quarter's earnings report. Output only the prompt and nothing else.\n\nQuestion: {question}",
                }
            ],
        }
    ]

    response = client.messages.create(model="claude-opus-4-1", max_tokens=2048, messages=messages)

    return response.content[0].text


haiku_prompt = generate_haiku_prompt(QUESTION)
print(haiku_prompt)

Extract the following information from the Apple earnings report PDF for the quarter:
1. Apple's net sales for the quarter
2. Quarter-over-quarter change in net sales
3. Key product categories, services, or regions that contributed significantly to the change in net sales
4. Any explanations provided for the changes in net sales

Organize the extracted information in a clear, concise format focusing on the key data points and insights related to the change in net sales for the quarter.


## 5단계: PDF에서 정보 추출하기
이제 질문을 정의하고 서브에이전트 Haiku 모델로 PDF에서 정보를 추출합니다. 각 모델에서 나온 정보를 깔끔하게 정의된 XML 태그 집합으로 정리합니다.

In [107]:
def extract_info(pdf_path, haiku_prompt):
    base64_encoded_pngs = pdf_to_base64_pngs(pdf_path)

    messages = [
        {
            "role": "user",
            "content": [
                *[
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": base64_encoded_png,
                        },
                    }
                    for base64_encoded_png in base64_encoded_pngs
                ],
                {"type": "text", "text": haiku_prompt},
            ],
        }
    ]

    response = client.messages.create(model="claude-haiku-4-5", max_tokens=2048, messages=messages)

    return response.content[0].text, pdf_path


def process_pdf(pdf_path):
    return extract_info(pdf_path, haiku_prompt)


# Process the PDFs concurrently with Haiku sub-agent models
with ThreadPoolExecutor() as executor:
    extracted_info_list = list(executor.map(process_pdf, pdf_paths))

extracted_info = ""
# Display the extracted information from each model call
for info in extracted_info_list:
    extracted_info += (
        '<info quarter="' + info[1].split("/")[-1].split("_")[1] + '">' + info[0] + "</info>\n"
    )
print(extracted_info)

<info quarter="Q4">According to the condensed consolidated statements of operations, Apple's net sales changed as follows in the 2023 financial year:

Quarter Ended September 30, 2023:
- Total net sales were $89,498 million, up from $90,146 million in the prior year quarter.
- Product sales were $67,184 million and services sales were $22,314 million.

Key contributors to the changes:
- Product sales decreased from $70,958 million in the prior year quarter.
- Services sales increased from $19,188 million in the prior year quarter.

Overall, Apple's total net sales decreased slightly compared to the same quarter in the prior year, driven by a decline in product sales which was partially offset by growth in services sales.</info>
<info quarter="Q3">Based on the financial statements provided, Apple's net sales changed as follows between the quarters in the 2023 financial year:

- Net sales increased from $81,797 million in the three months ended July 1, 2023 to $82,959 million in the thre

서브에이전트 모델들로 PDF에서 정보를 동시에 추출한 뒤 합칩니다. 그런 다음 질문과 추출된 정보를 담아 강력한 모델에 보낼 메시지를 준비하고, 답변과 matplotlib 코드를 생성해 달라고 요청합니다.

## 6단계: 정보를 Opus에 전달해 답변 생성하기
서브에이전트로 각 PDF에서 정보를 가져왔으니, 이제 Opus를 호출해 실제로 질문에 답하고 답변에 곁들일 그래프를 만드는 코드를 작성하게 합니다.

In [108]:
# Prepare the messages for the powerful model
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": f"Based on the following extracted information from Apple's earnings releases, please provide a response to the question: {QUESTION}\n\nAlso, please generate Python code using the matplotlib library to accompany your response. Enclose the code within <code> tags.\n\nExtracted Information:\n{extracted_info}",
            }
        ],
    }
]

# Generate the matplotlib code using the powerful model
response = client.messages.create(model="claude-opus-4-1", max_tokens=4096, messages=messages)

generated_response = response.content[0].text
print("Generated Response:")
print(generated_response)

Generated Response:
Based on the extracted information from Apple's earnings releases, Apple's net sales changed as follows in the 2023 financial year:

In Q1, net sales increased from $117,154 million in the previous quarter to $123,945 million, driven by increases in both product sales and services revenue.

In Q2, net sales decreased by around $2,442 million compared to the prior six-month period, primarily due to a decrease in product sales, which was partially offset by an increase in services sales.

In Q3, net sales increased by approximately $1,162 million compared to the previous quarter, with growth in both product sales and services sales contributing to the overall increase.

In Q4, total net sales decreased slightly compared to the same quarter in the prior year, driven by a decline in product sales, which was partially offset by growth in services sales.

Here's a Python code snippet using the matplotlib library to visualize the quarterly net sales data:

<code>
import ma

## 7단계: 응답 추출하고 Matplotlib 코드 실행하기
마지막으로 생성된 응답에서 matplotlib 코드를 추출해 실행하고 매출 성장 추이를 시각화합니다.

```extract_code_and_response``` 함수를 정의해 생성된 응답에서 matplotlib 코드와 코드가 아닌 부분을 분리합니다. 코드가 아닌 응답을 출력하고, matplotlib 코드가 있으면 실행합니다.

참고: 모델이 작성한 코드를 샌드박스 밖에서 ```exec```로 실행하는 것은 좋은 관행이 아니지만, 이 데모에서는 그렇게 하고 있습니다 :)

In [ ]:
# Extract the matplotlib code from the response
# Function to extract the code and non-code parts from the response
def extract_code_and_response(response):
    start_tag = "<code>"
    end_tag = "</code>"
    start_index = response.find(start_tag)
    end_index = response.find(end_tag)
    if start_index != -1 and end_index != -1:
        code = response[start_index + len(start_tag) : end_index].strip()
        non_code_response = response[:start_index].strip()
        return code, non_code_response
    else:
        return None, response.strip()


matplotlib_code, non_code_response = extract_code_and_response(generated_response)

print(non_code_response)
if matplotlib_code:
    # Execute the extracted matplotlib code
    # Note: exec is used here for demonstration purposes to run model-generated visualization code.
    # In production, use a sandboxed environment for executing untrusted code.
    exec(matplotlib_code)  # noqa: S102
else:
    print("No matplotlib code found in the response.")